In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q -U \
  transformers==4.45.0 \
  peft==0.13.0 \
  trl==0.8.6 \
  bitsandbytes==0.49.2 \
  accelerate \
  datasets \
  evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.2 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset("glue", "sst2")

print(dataset)
print("\n--- Sample Example ---")
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

--- Sample Example ---
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [ ]:
def format_prompt(sample):
    label_text = "positive" if sample["label"] == 1 else "negative"
    return {
        "formatted": f"""### Instruction:
Classify the sentiment of the following movie review as positive or negative.

### Review:
{sample["sentence"]}

### Sentiment:
{label_text}"""
    }

# Use 1000 samples for training, 200 for evaluation (faster on T4)
train_data = dataset["train"].select(range(1000)).map(format_prompt)
eval_data  = dataset["validation"].select(range(200)).map(format_prompt)

print(f"Train samples : {len(train_data)}")
print(f"Eval samples  : {len(eval_data)}")
print("\n--- Formatted Sample ---")
print(train_data[0]["formatted"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Train samples : 1000
Eval samples  : 200

--- Formatted Sample ---
### Instruction:
Classify the sentiment of the following movie review as positive or negative.

### Review:
hide new secretions from the parental units 

### Sentiment:
negative


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

print("✅ TinyLlama 1.1B loaded successfully!")
print(f"Device: {next(model.parameters()).device}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ TinyLlama 1.1B loaded successfully!
Device: cuda:0


In [ ]:
def generate_sentiment(sentence):
    input_text = f"""### Instruction:
Classify the sentiment of the following movie review as positive or negative.

### Review:
{sentence}

### Sentiment:
"""
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Sentiment:" in generated:
        sentiment = generated.split("### Sentiment:")[-1].strip().lower()
    else:
        sentiment = generated.strip().lower()

    # Extract just positive or negative
    if "positive" in sentiment:
        return "positive"
    elif "negative" in sentiment:
        return "negative"
    else:
        return "unknown"


def evaluate_accuracy(data, num_samples=200):
    correct = 0
    total = 0
    unknown = 0

    for i, sample in enumerate(data):
        if i >= num_samples:
            break

        predicted = generate_sentiment(sample["sentence"])
        actual = "positive" if sample["label"] == 1 else "negative"

        if predicted == "unknown":
            unknown += 1
        elif predicted == actual:
            correct += 1

        total += 1

        if (i + 1) % 40 == 0:
            print(f"Progress: {i+1}/{num_samples} | Correct: {correct} | Unknown: {unknown}")

    accuracy = correct / total * 100
    print(f"\n✅ Baseline Accuracy: {correct}/{total} = {accuracy:.2f}%")
    print(f"   Unknown responses : {unknown}")
    return accuracy

baseline_score = evaluate_accuracy(eval_data, num_samples=200)

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Progress: 40/200 | Correct: 20 | Unknown: 0
Progress: 80/200 | Correct: 43 | Unknown: 0
Progress: 120/200 | Correct: 65 | Unknown: 0
Progress: 160/200 | Correct: 85 | Unknown: 0
Progress: 200/200 | Correct: 104 | Unknown: 0

✅ Baseline Accuracy: 104/200 = 52.00%
   Unknown responses : 0


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training arguments
training_args = TrainingArguments(
    output_dir="./tinyllama-sentiment",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none"
)

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    dataset_text_field="formatted",
    max_seq_length=256,
    tokenizer=tokenizer,
)

print("✅ Trainer ready!")

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Trainer ready!


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,2.229000
40,1.045400
60,0.932400
80,0.887900
100,0.881600
120,0.861800
140,0.895600
160,0.802300
180,0.835600
200,0.831900


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=310, training_loss=0.932917038086922, metrics={'train_runtime': 463.7238, 'train_samples_per_second': 10.782, 'train_steps_per_second': 0.669, 'total_flos': 1940353616904192.0, 'train_loss': 0.932917038086922, 'epoch': 4.96})

In [ ]:
model.save_pretrained("./tinyllama-sentiment")
tokenizer.save_pretrained("./tinyllama-sentiment")

print("✅ Fine-tuned model saved!")

✅ Fine-tuned model saved!


In [ ]:
finetuned_score = evaluate_accuracy(eval_data, num_samples=200)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Progress: 40/200 | Correct: 39 | Unknown: 0
Progress: 80/200 | Correct: 79 | Unknown: 0
Progress: 120/200 | Correct: 115 | Unknown: 0
Progress: 160/200 | Correct: 155 | Unknown: 0
Progress: 200/200 | Correct: 192 | Unknown: 0

✅ Baseline Accuracy: 192/200 = 96.00%
   Unknown responses : 0


In [ ]:
print("=" * 40)
print("   FINE-TUNING RESULTS SUMMARY")
print("=" * 40)
print(f"  Baseline Accuracy  : 52.00%")
print(f"  Fine-tuned Accuracy: 96.00%")
print(f"  Improvement        : +44.00%")
print("=" * 40)
print("  Model  : TinyLlama 1.1B")
print("  Task   : Sentiment Analysis (SST-2)")
print("  Method : QLoRA Fine-tuning")
print("  Train  : 1000 samples, 5 epochs")
print("=" * 40)

   FINE-TUNING RESULTS SUMMARY
  Baseline Accuracy  : 52.00%
  Fine-tuned Accuracy: 96.00%
  Improvement        : +44.00%
  Model  : TinyLlama 1.1B
  Task   : Sentiment Analysis (SST-2)
  Method : QLoRA Fine-tuning
  Train  : 1000 samples, 5 epochs
